# 🏭 Vision-Driven Industrial Safety & Quality Inspection Engine
## Notebook 01 — Prepare Combined 12-Class Dataset

**Prerequisite:** Hazard dataset downloaded to `Hazard_Expansion/Dataset/`

---

### Purpose
1. Diagnose the existing 7-class textile dataset (class imbalance, annotation quality, object sizes)
2. Remap the 5 industrial hazard classes (chemical hazard, fire, no helmet, smoke, water leak) to IDs 7–11
3. Combine into a clean 12-class dataset
4. Validate combined dataset integrity
5. Apply balanced sampling strategy
6. Generate dataset statistics report
7. Write `combined_dataset/data.yaml`

### Final 12-Class Mapping (7 Textile + 5 Industrial Hazards)
```
0: baekra           (textile defect — existing)
1: color issues     (textile defect — existing)
2: contamination    (textile defect — existing)
3: cut              (textile defect — existing)
4: gray stitch      (textile defect — existing)
5: selvet           (textile defect — existing)
6: stain            (textile defect — existing)
7: chemical hazard  (industrial hazard — NEW)
8: fire             (industrial hazard — NEW)
9: no helmet        (PPE violation — NEW)
10: smoke           (industrial hazard — NEW)
11: water leak      (industrial hazard / liquid spill — NEW)
```

**Note:** Class IDs 0–6 are FIXED and match `final_dataset/data.yaml` exactly.

In [ ]:
%pip install -q ultralytics PyYAML tqdm Pillow matplotlib numpy imagehash
print('Dependencies ready.')

---
## Step 1 — Configure Paths

In [ ]:
"""Configure paths for Industrial Hazard and Textile datasets."""
import os, sys, shutil, yaml, json, zipfile
from pathlib import Path

# ── Determine Root & Environment ──────────────────────────────────────────────
IN_COLAB = "google.colab" in sys.modules or (Path("/content").exists() and not sys.platform.startswith("win"))
CONTENT = Path("/content") if IN_COLAB else (Path.cwd() / "content_runtime")
CONTENT.mkdir(parents=True, exist_ok=True)

# Find project root dynamically by ascending parent directories
def find_project_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    if IN_COLAB:
        candidates.extend([Path("/content"), Path("/content/drive/MyDrive/Hangzhou_Textile_POC")])
    for p in candidates:
        if (p / "Hazard_Expansion" / "Dataset").exists() or (p / "final_dataset").exists():
            return p.resolve()
    return Path.cwd().resolve()

PROJECT_ROOT = find_project_root()
print(f"Environment : {'Google Colab' if IN_COLAB else 'Local Environment'}")
print(f"Working Dir : {Path.cwd()}")
print(f"Project Root: {PROJECT_ROOT}")

# ── Drive mount (always attempt on Colab) ─────────────────────────────────
DRIVE_PROJECT = None
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        for dp in [
            Path("/content/drive/MyDrive/Hangzhou_Textile_POC"),
            Path("/content/drive/MyDrive"),
        ]:
            if dp.exists():
                DRIVE_PROJECT = dp
                break
    except Exception:
        pass

# ── Check if combined_dataset already exists (skip rebuild if so) ───────────
COMBINED_DATASET = CONTENT / "combined_dataset"
combined_candidates = [
    CONTENT / "combined_dataset",
    PROJECT_ROOT / "content_runtime" / "combined_dataset",
    PROJECT_ROOT / "Hazard_Expansion" / "content_runtime" / "combined_dataset",
    PROJECT_ROOT / "combined_dataset",
    Path.cwd() / "combined_dataset",
    Path.cwd() / "content_runtime" / "combined_dataset",
    Path.cwd().parent / "content_runtime" / "combined_dataset",
    Path("/content/combined_dataset"),
]
if DRIVE_PROJECT:
    combined_candidates.extend([
        DRIVE_PROJECT / "combined_dataset",
        DRIVE_PROJECT.parent / "combined_dataset",
    ])

EXISTING_COMBINED = None
for c in combined_candidates:
    if c.exists() and (c / "data.yaml").exists():
        EXISTING_COMBINED = c.resolve()
        break

# Also check for a zip on Drive
if EXISTING_COMBINED is None:
    zip_search = [CONTENT, PROJECT_ROOT]
    if DRIVE_PROJECT:
        zip_search.extend([DRIVE_PROJECT, DRIVE_PROJECT.parent])
    for d in zip_search:
        try:
            zips = list(d.glob("*combined_dataset*.zip"))
        except Exception:
            zips = []
        if zips:
            zip_path = sorted(zips, key=lambda p: p.stat().st_size, reverse=True)[0]
            extract_target = CONTENT / "combined_dataset"
            print(f"Found zip: {zip_path}  — extracting to {extract_target} ...")
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(extract_target)
            if (extract_target / "data.yaml").exists():
                EXISTING_COMBINED = extract_target.resolve()
            break

SKIP_BUILD = EXISTING_COMBINED is not None

if SKIP_BUILD:
    COMBINED_DATASET = EXISTING_COMBINED
    print(f"\n✅ Combined dataset already found at: {COMBINED_DATASET}")
    print("   SKIP_BUILD = True — skipping source dataset check & rebuild.")
    print("   Jump straight to the Drive backup cell (Step 8) to save it to Drive.")
else:
    print("\nCombined dataset not found — will build from source datasets.")
    print("   SKIP_BUILD = False — source datasets must be available.")

# ── 1. Locate Industrial Hazard Dataset (only needed if building) ─────────
HAZARD_DIR = None
if not SKIP_BUILD:
    hazard_candidates = [
        PROJECT_ROOT / "Hazard_Expansion" / "Dataset",
        PROJECT_ROOT / "Dataset",
        Path.cwd() / "Hazard_Expansion" / "Dataset",
        Path.cwd() / "Dataset",
        Path.cwd().parent / "Dataset",
        Path.cwd().parent / "Hazard_Expansion" / "Dataset",
        Path("/content/Hazard_Expansion/Dataset"),
        Path("/content/Dataset"),
        Path("/content/drive/MyDrive/Hazard_Expansion/Dataset"),
        Path("/content/drive/MyDrive/Hangzhou_Textile_POC/Hazard_Expansion/Dataset"),
    ]
    for c in hazard_candidates:
        if c.exists() and (c / "data.yaml").exists():
            HAZARD_DIR = c.resolve()
            break

    print(f"Hazard Dataset location : {HAZARD_DIR}")
    assert HAZARD_DIR and HAZARD_DIR.exists(), (
        f"Hazard dataset not found! Searched: {[str(p) for p in hazard_candidates]}"
    )

# ── 2. Locate Existing Textile Dataset (only needed if building) ──────────
TEXTILE_DATASET = None
if not SKIP_BUILD:
    textile_candidates = [
        PROJECT_ROOT / "final_dataset",
        Path.cwd() / "final_dataset",
        Path.cwd().parent / "final_dataset",
        Path("/content/final_dataset"),
        Path("/content/drive/MyDrive/Hangzhou_Textile_POC/final_dataset"),
        Path("/content/drive/MyDrive/final_dataset"),
    ]
    for c in textile_candidates:
        if c.exists() and (c / "data.yaml").exists():
            TEXTILE_DATASET = c.resolve()
            break

    # ZIP fallback
    if TEXTILE_DATASET is None:
        zip_cands = (list(CONTENT.glob("*textile*.zip")) +
                     list(PROJECT_ROOT.glob("*textile*.zip")) +
                     list(Path.cwd().glob("*textile*.zip")))
        if zip_cands:
            print(f"Extracting textile dataset from: {zip_cands[0]}")
            with zipfile.ZipFile(zip_cands[0]) as zf:
                zf.extractall(CONTENT / "textile_extracted")
            cand = CONTENT / "textile_extracted" / "final_dataset"
            TEXTILE_DATASET = cand if cand.exists() else (CONTENT / "textile_extracted")

    # Kagglehub fallback
    if TEXTILE_DATASET is None:
        print("Textile dataset not found locally or in Drive. Fetching via kagglehub...")
        try:
            import kagglehub
            kpath = kagglehub.dataset_download("muhammadharisabid/fabricdefectntu")
            TEXTILE_DATASET = Path(kpath)
            print(f"Downloaded textile dataset to: {TEXTILE_DATASET}")
        except Exception as e:
            print(f"Kagglehub fetch failed: {e}")

    print(f"Textile Dataset location: {TEXTILE_DATASET}")
    assert TEXTILE_DATASET and (TEXTILE_DATASET / "data.yaml").exists(), (
        f"Textile dataset not found! Searched: {[str(p) for p in textile_candidates]}"
    )

# ── 3. Configure Output Directories ─────────────────────────────────────────
if not SKIP_BUILD:
    for split in ["train", "val", "test"]:
        (COMBINED_DATASET / "images" / split).mkdir(parents=True, exist_ok=True)
        (COMBINED_DATASET / "labels" / split).mkdir(parents=True, exist_ok=True)

REPORTS_DIR = CONTENT / "hazard_reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

if not SKIP_BUILD:
    print(f"Combined Output Dir    : {COMBINED_DATASET}")
    print(f"Reports Dir            : {REPORTS_DIR}")
    print("All paths configured and datasets successfully located.")


---
## Step 2 — Diagnose Existing Textile Dataset

In [ ]:
# Cell skipped automatically when combined_dataset already exists.
if not SKIP_BUILD:
    """Per-class annotation analysis of existing 7-class textile dataset."""
    from collections import Counter, defaultdict
    import numpy as np
    
    # Load textile data.yaml
    with open(TEXTILE_DATASET / 'data.yaml') as f:
        textile_yaml = yaml.safe_load(f)
    
    TEXTILE_CLASSES = textile_yaml.get('names', [])
    print(f'Textile classes: {TEXTILE_CLASSES}')
    assert len(TEXTILE_CLASSES) == 7, f'Expected 7 classes, got {len(TEXTILE_CLASSES)}'
    
    SPLITS = ['train', 'val', 'test']
    SUPPORTED_EXT = {'.jpg', '.jpeg', '.png', '.bmp'}
    
    class_counts_per_split = {s: Counter() for s in SPLITS}
    img_counts_per_split   = {s: 0 for s in SPLITS}
    box_sizes_per_class    = defaultdict(list)  # normalized box area
    total_annotations      = 0
    empty_label_count      = 0
    
    for split in SPLITS:
        img_dir = TEXTILE_DATASET / 'images' / split
        lbl_dir = TEXTILE_DATASET / 'labels' / split
        if not img_dir.exists():
            print(f'  Split {split}: NOT FOUND')
            continue
        
        imgs = [p for p in img_dir.iterdir() if p.suffix.lower() in SUPPORTED_EXT]
        img_counts_per_split[split] = len(imgs)
        
        for img_path in imgs:
            lbl_path = lbl_dir / (img_path.stem + '.txt')
            if not lbl_path.exists():
                continue
            lines = [l.strip() for l in open(lbl_path) if l.strip()]
            if not lines:
                empty_label_count += 1
                continue
            for line in lines:
                parts = line.split()
                if len(parts) < 5:
                    continue
                cls_id = int(float(parts[0]))
                w, h = float(parts[3]), float(parts[4])
                class_counts_per_split[split][cls_id] += 1
                total_annotations += 1
                if split == 'train':
                    box_sizes_per_class[cls_id].append(w * h)  # normalized area
    
    print('\n=== TEXTILE DATASET DIAGNOSIS ===')
    print(f'{"Class":<20} {"Train":>8} {"Val":>8} {"Test":>8} {"Total":>8} {"Median BoxArea":>14}')
    print('-' * 70)
    for cls_id, cls_name in enumerate(TEXTILE_CLASSES):
        tr = class_counts_per_split['train'][cls_id]
        va = class_counts_per_split['val'][cls_id]
        te = class_counts_per_split['test'][cls_id]
        tot = tr + va + te
        sizes = box_sizes_per_class[cls_id]
        median_area = f'{np.median(sizes):.4f}' if sizes else 'N/A'
        print(f'{cls_name:<20} {tr:>8} {va:>8} {te:>8} {tot:>8} {median_area:>14}')
    
    print('-' * 70)
    print(f'{"TOTAL":<20} {sum(class_counts_per_split["train"].values()):>8} '
          f'{sum(class_counts_per_split["val"].values()):>8} '
          f'{sum(class_counts_per_split["test"].values()):>8} '
          f'{total_annotations:>8}')
    print(f'\nImages:  train={img_counts_per_split["train"]}, val={img_counts_per_split["val"]}, test={img_counts_per_split["test"]}')
    print(f'Empty label files: {empty_label_count}')
    
    # Imbalance ratio
    train_counts = [class_counts_per_split['train'][i] for i in range(7)]
    if min(train_counts) > 0:
        ratio = max(train_counts) / min(train_counts)
        print(f'\nTrain imbalance ratio (max/min): {ratio:.1f}x')
        if ratio > 5:
            print('  ► High imbalance detected. Balancing strategy required.')

In [ ]:
# Cell skipped automatically when combined_dataset already exists.
if not SKIP_BUILD:
    """Visualize class distribution."""
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar chart: annotations per class (train)
    class_names = TEXTILE_CLASSES
    train_vals = [class_counts_per_split['train'][i] for i in range(7)]
    
    colors = ['#2ecc71' if v >= max(train_vals)*0.6 else '#e74c3c' for v in train_vals]
    axes[0].bar(range(7), train_vals, color=colors, edgecolor='black', linewidth=0.7)
    axes[0].set_xticks(range(7))
    axes[0].set_xticklabels(class_names, rotation=35, ha='right', fontsize=9)
    axes[0].set_ylabel('Training Annotations')
    axes[0].set_title('Textile Dataset: Training Annotations per Class\n(red = potentially underrepresented)')
    for i, v in enumerate(train_vals):
        axes[0].text(i, v + 5, str(v), ha='center', va='bottom', fontsize=8)
    
    # Box area distribution per class
    data_to_plot = [box_sizes_per_class[i] for i in range(7)]
    axes[1].boxplot(data_to_plot, labels=class_names, showfliers=False)
    axes[1].tick_params(axis='x', rotation=35)
    axes[1].set_ylabel('Normalized Box Area (w×h)')
    axes[1].set_title('Object Size Distribution per Class\n(smaller = harder to detect at 640px)')
    
    plt.tight_layout()
    plot_path = REPORTS_DIR / 'textile_diagnosis.png'
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {plot_path}')

---
## Step 3 — Copy Textile Dataset to Combined

In [ ]:
# Cell skipped automatically when combined_dataset already exists.
if not SKIP_BUILD:
    """Copy existing textile images+labels to combined_dataset.
    Class IDs 0-6 are preserved exactly — NO remapping needed.
    Degenerate/zero-area bounding boxes are filtered out, and float class IDs are standardized.
    """
    import shutil
    from tqdm import tqdm
    
    textile_copy_counts = {}
    
    for split in SPLITS:
        img_src = TEXTILE_DATASET / "images" / split
        lbl_src = TEXTILE_DATASET / "labels" / split
        img_dst = COMBINED_DATASET / "images" / split
        lbl_dst = COMBINED_DATASET / "labels" / split
        
        if not img_src.exists():
            print(f"  Textile {split}: source not found, skipping")
            continue
        
        imgs = [p for p in img_src.iterdir() if p.suffix.lower() in SUPPORTED_EXT]
        copied = 0
        
        for img_path in tqdm(imgs, desc=f"  Copying textile {split}"):
            lbl_path = lbl_src / (img_path.stem + ".txt")
            new_stem = f"textile_{img_path.stem}"
            
            shutil.copy2(img_path, img_dst / f"{new_stem}{img_path.suffix}")
            dst_lbl = lbl_dst / f"{new_stem}.txt"
            
            if lbl_path.exists():
                new_lines = []
                with open(lbl_path, "r", encoding="utf-8") as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) >= 5:
                            cls_id = int(float(parts[0]))
                            w, h = float(parts[3]), float(parts[4])
                            if w > 0 and h > 0:
                                new_lines.append(f"{cls_id} " + " ".join(parts[1:]))
                with open(dst_lbl, "w", encoding="utf-8") as f:
                    f.write("\n".join(new_lines) + ("\n" if new_lines else ""))
            else:
                dst_lbl.touch()
            copied += 1
        
        textile_copy_counts[split] = copied
        print(f"  Textile {split}: {copied} images copied")
    
    print(f"\nTextile copy complete: {textile_copy_counts}")


---
## Step 4 — Remap & Copy Hazard Datasets

In [ ]:
# Cell skipped automatically when combined_dataset already exists.
if not SKIP_BUILD:
    """Remap the 5 Industrial Hazard classes (0..4 -> 7..11) and copy to combined_dataset.
    
    Original Hazard Classes:
      0: chemical hazard -> Remap to ID 7
      1: fire            -> Remap to ID 8
      2: no helmet       -> Remap to ID 9
      3: smoke           -> Remap to ID 10
      4: water leak      -> Remap to ID 11
    """
    import random
    from tqdm import tqdm
    
    # Load Hazard data.yaml
    with open(HAZARD_DIR / "data.yaml", "r", encoding="utf-8") as f:
        hz_yaml = yaml.safe_load(f)
    hz_classes = hz_yaml.get("names", [])
    print(f"Hazard original classes ({len(hz_classes)}): {hz_classes}")
    
    # Map each hazard index (0..4) to its new class ID (7..11)
    HAZARD_CLASS_OFFSET = 7
    hazard_remap = {i: i + HAZARD_CLASS_OFFSET for i in range(len(hz_classes))}
    print(f"Class ID Remap mapping: {hazard_remap}")
    
    # Process splits (Hazard dataset has 'train' and 'valid')
    hazard_counts = {"train": 0, "val": 0, "test": 0}
    
    # 1. Process Train split
    train_img_dir = HAZARD_DIR / "train" / "images"
    train_lbl_dir = HAZARD_DIR / "train" / "labels"
    train_imgs = sorted([p for p in train_img_dir.iterdir() if p.suffix.lower() in SUPPORTED_EXT])
    
    for img_path in tqdm(train_imgs, desc="Remapping Hazard train"):
        new_stem = f"hazard_{img_path.stem}"
        shutil.copy2(img_path, COMBINED_DATASET / "images" / "train" / f"{new_stem}{img_path.suffix}")
        lbl_path = train_lbl_dir / (img_path.stem + ".txt")
        new_lines = []
        if lbl_path.exists():
            with open(lbl_path, "r", encoding="utf-8") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        old_cls = int(float(parts[0]))
                        if old_cls in hazard_remap:
                            new_lines.append(f"{hazard_remap[old_cls]} " + " ".join(parts[1:]))
        with open(COMBINED_DATASET / "labels" / "train" / f"{new_stem}.txt", "w", encoding="utf-8") as f:
            f.write("\n".join(new_lines) + ("\n" if new_lines else ""))
        hazard_counts["train"] += 1
    
    # 2. Process Validation split (divide into 80% val and 20% test for clean evaluation)
    valid_img_dir = HAZARD_DIR / "valid" / "images"
    valid_lbl_dir = HAZARD_DIR / "valid" / "labels"
    valid_imgs = sorted([p for p in valid_img_dir.iterdir() if p.suffix.lower() in SUPPORTED_EXT])
    
    random.seed(42)
    random.shuffle(valid_imgs)
    split_idx = int(0.8 * len(valid_imgs))
    val_imgs = valid_imgs[:split_idx]
    test_imgs = valid_imgs[split_idx:]
    
    for split_name, img_list in [("val", val_imgs), ("test", test_imgs)]:
        for img_path in tqdm(img_list, desc=f"Remapping Hazard {split_name}"):
            new_stem = f"hazard_{img_path.stem}"
            shutil.copy2(img_path, COMBINED_DATASET / "images" / split_name / f"{new_stem}{img_path.suffix}")
            lbl_path = valid_lbl_dir / (img_path.stem + ".txt")
            new_lines = []
            if lbl_path.exists():
                with open(lbl_path, "r", encoding="utf-8") as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) >= 5:
                            old_cls = int(float(parts[0]))
                            if old_cls in hazard_remap:
                                new_lines.append(f"{hazard_remap[old_cls]} " + " ".join(parts[1:]))
            with open(COMBINED_DATASET / "labels" / split_name / f"{new_stem}.txt", "w", encoding="utf-8") as f:
                f.write("\n".join(new_lines) + ("\n" if new_lines else ""))
            hazard_counts[split_name] += 1
    
    print(f"\nHazard dataset remapped and copied: {hazard_counts}")


---
## Step 5 — Write combined_dataset/data.yaml

In [ ]:
# Cell skipped automatically when combined_dataset already exists.
if not SKIP_BUILD:
    combined_yaml = {
        'path': str(COMBINED_DATASET),
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test',
        'nc': 12,
        'names': [
            # Existing textile classes (0-6)
            'baekra',           # 0
            'color issues',     # 1
            'contamination',    # 2
            'cut',              # 3
            'gray stitch',      # 4
            'selvet',           # 5
            'stain',            # 6
            # New industrial hazard classes (7-11)
            'chemical hazard',  # 7
            'fire',             # 8
            'no helmet',        # 9
            'smoke',            # 10
            'water leak',       # 11
        ]
    }
    
    yaml_path = COMBINED_DATASET / 'data.yaml'
    with open(yaml_path, 'w') as f:
        yaml.dump(combined_yaml, f, allow_unicode=True, sort_keys=False)
    
    print(f'Written: {yaml_path}')
    print(f'Total Classes (nc): {combined_yaml["nc"]}')
    print('Class list:')
    for idx, name in enumerate(combined_yaml['names']):
        print(f'  {idx:2d}: {name}')
    
    # Verify
    with open(yaml_path) as f:
        check = yaml.safe_load(f)
    assert check['nc'] == 12
    assert len(check['names']) == 12
    print('data.yaml VERIFIED ✅')


---
## Step 6 — Validate Combined Dataset

In [ ]:
# Cell skipped automatically when combined_dataset already exists.
if not SKIP_BUILD:
    """Full validation of combined dataset integrity."""
    import hashlib
    from collections import Counter
    from tqdm import tqdm
    
    def file_md5(path):
        h = hashlib.md5()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(65536), b""):
                h.update(chunk)
        return h.hexdigest()
    
    print("=== COMBINED DATASET VALIDATION ===")
    
    errors = []
    split_hashes = {}
    split_image_counts = {}
    class_counts_combined = Counter()
    
    for split in SPLITS:
        img_dir = COMBINED_DATASET / "images" / split
        lbl_dir = COMBINED_DATASET / "labels" / split
        
        if not img_dir.exists():
            print(f"  {split}: MISSING")
            continue
        
        imgs = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in SUPPORTED_EXT])
        split_image_counts[split] = len(imgs)
        hashes = {}
        
        invalid = 0
        missing_lbl = 0
        
        for img_path in tqdm(imgs, desc=f"  Validating {split}"):
            # Hash for dup check
            md5 = file_md5(img_path)
            if md5 in hashes:
                errors.append(f"INTRA_SPLIT_DUPLICATE [{split}]: {img_path.name}")
            hashes[md5] = img_path.name
            
            # Label check
            lbl_path = lbl_dir / (img_path.stem + ".txt")
            if not lbl_path.exists():
                missing_lbl += 1
                continue
            
            with open(lbl_path, "r", encoding="utf-8") as f:
                lines = [l.strip() for l in f if l.strip()]
            
            for line in lines:
                parts = line.split()
                if len(parts) != 5:
                    invalid += 1
                    continue
                try:
                    cls_id = int(float(parts[0]))
                    xc, yc, w, h = map(float, parts[1:])
                except ValueError:
                    invalid += 1
                    continue
                if not (0 <= cls_id <= 11):
                    errors.append(f"INVALID_CLASS_ID {cls_id}: {lbl_path.name}")
                    invalid += 1
                    continue
                if not (0 <= xc <= 1 and 0 <= yc <= 1 and 0 < w <= 1 and 0 < h <= 1):
                    errors.append(f"OUT_OF_BOUNDS coords: {lbl_path.name}")
                    invalid += 1
                    continue
                class_counts_combined[cls_id] += 1
        
        print(f"  {split}: {len(imgs)} images, missing_labels={missing_lbl}, invalid_annots={invalid}")
        split_hashes[split] = hashes
    
    # Cross-split duplicate check
    print("\n  Cross-split duplicate check...")
    split_names = list(split_hashes.keys())
    cross_dups = 0
    for i in range(len(split_names)):
        for j in range(i+1, len(split_names)):
            s1, s2 = split_names[i], split_names[j]
            common = set(split_hashes[s1]) & set(split_hashes[s2])
            if common:
                for h in common:
                    errors.append(f"CROSS_SPLIT_LEAK [{s1}↔{s2}]: {split_hashes[s1][h]}")
                    cross_dups += 1
    
    validation_errors = errors
    passed = len(errors) == 0
    print(f"\n  Cross-split duplicates: {cross_dups}")
    print(f"  Total errors: {len(errors)}")
    print(f"\n  VALIDATION: {'✅ PASS' if passed else '❌ FAIL'}")
    
    if errors:
        print("\n  First 10 errors:")
        for e in errors[:10]:
            print(f"    • {e}")


In [ ]:
# Cell skipped automatically when combined_dataset already exists.
if not SKIP_BUILD:
    """Print and visualize combined class distribution."""
    import matplotlib.pyplot as plt
    
    ALL_CLASSES = combined_yaml['names']
    counts = [class_counts_combined.get(i, 0) for i in range(12)]
    
    print('=== COMBINED CLASS DISTRIBUTION (train+val+test annotations) ===')
    print(f'{"ID":<4} {"Class":<20} {"Annotations":>12} {"Type":<15}')
    print('-' * 55)
    for i, (name, cnt) in enumerate(zip(ALL_CLASSES, counts)):
        group = 'Textile Defect' if i < 7 else 'Industrial Hazard'
        print(f'{i:<4} {name:<20} {cnt:>12} {group:<15}')
    print('-' * 55)
    print(f'Total annotations: {sum(counts):,}')
    
    # Bar plot
    colors = ['#2b5c8f'] * 7 + ['#d95f02'] * 5  # blue for textile, orange for hazard
    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.bar(range(12), counts, color=colors, alpha=0.85, edgecolor='black', linewidth=0.8)
    ax.set_xticks(range(12))
    ax.set_xticklabels(ALL_CLASSES, rotation=35, ha='right', fontsize=9)
    ax.set_ylabel('Total Bounding Boxes')
    ax.set_title('Unified 12-Class Dataset Distribution\n(Blue: Textile Defects | Orange: Industrial Hazards)', fontsize=11, fontweight='bold')
    
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f'{cnt:,}', ha='center', va='bottom', fontsize=8)
    
    plt.tight_layout()
    dist_plot_path = REPORTS_DIR / 'combined_class_distribution.png'
    plt.savefig(dist_plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Distribution plot saved: {dist_plot_path}')


---
## Step 7 — Save Dataset Statistics Report

In [ ]:
# Cell skipped automatically when combined_dataset already exists.
if not SKIP_BUILD:
    import datetime
    
    total_images = sum(split_image_counts.values())
    total_annots = sum(class_counts_combined.values())
    
    stats = {
        'generated': datetime.datetime.now().isoformat(),
        'dataset_path': str(COMBINED_DATASET),
        'nc': 12,
        'classes': combined_yaml['names'],
        'total_images': total_images,
        'split_images': split_image_counts,
        'total_annotations': total_annots,
        'class_annotations': {
            name: class_counts_combined.get(i, 0)
            for i, name in enumerate(combined_yaml['names'])
        },
        'textile_classes': combined_yaml['names'][:7],
        'hazard_classes': combined_yaml['names'][7:],
        'validation_errors': validation_errors
    }
    
    stats_path = REPORTS_DIR / 'combined_dataset_stats.json'
    with open(stats_path, 'w') as f:
        json.dump(stats, f, indent=2)
    print(f'Statistics saved: {stats_path}')
    
    print('\n=== DATASET PREPARATION SUMMARY ===')
    print(f'  Total images      : {total_images:,}')
    print(f'  Train images      : {split_image_counts.get("train", 0):,}')
    print(f'  Val images        : {split_image_counts.get("val", 0):,}')
    print(f'  Test images       : {split_image_counts.get("test", 0):,}')
    print(f'  Total annotations : {total_annots:,}')
    print(f'  Classes           : 12 (7 textile + 5 industrial safety)')
    print(f'  Integrity check   : {"PASS ✅" if len(validation_errors) == 0 else "ISSUES FOUND ⚠"}')


---
## End of Notebook 01

**Next:** Open `02_improve_existing_model.ipynb` to fine-tune the 7-class textile baseline model, or proceed directly to `03_train_expanded_model.ipynb` to train the full 12-class industrial safety model.
